# Local-to-particle evaluation (L2P)

## Purpose

L2P evaluates a target-centred local Taylor expansion at particle positions.
It is the final far-field operator in a conventional FMM. We first build a
valid local expansion with P2M and M2L, then map its spatial error across a
plane around the local centre.

## Mathematical definition

For $dx=x-c_t$,

$$\phi_p(x)=\sum_{|\beta|\le p}L_\beta\frac{dx^\beta}{\beta!},
\qquad H_p(x)=-\nabla\phi_p(x).$$

The local expansion is most accurate near $c_t$. Its valid region depends on
the separation from the source cluster and on truncation order.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import cdfmm

try:
    from example_utils import (
        direct_fields,
        draw_box_3d,
        error_metrics,
        finish_3d_axes,
        local_fields,
        multipole_fields,
        new_3d_figure,
        nodes_at_level,
        plot_coefficients_by_degree,
        random_unit_vectors,
        relative_error,
        set_axes_equal,
        vec3_to_array,
    )
except ModuleNotFoundError:
    # This path is used when the kernel starts in the repository root.
    from examples.notebooks.example_utils import (
        direct_fields,
        draw_box_3d,
        error_metrics,
        finish_3d_axes,
        local_fields,
        multipole_fields,
        new_3d_figure,
        nodes_at_level,
        plot_coefficients_by_degree,
        random_unit_vectors,
        relative_error,
        set_axes_equal,
        vec3_to_array,
    )

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True})

## User parameters

In [ ]:
expansion_order = 5
n_sources = 80
random_seed = 42
source_centre = np.zeros(3)
local_centre = np.array([4.0, 0.5, 0.0])
plane_half_width = 0.45
grid_points_per_axis = 23

## Build the local expansion

In [ ]:
rng = np.random.default_rng(random_seed)
source_positions = rng.uniform(-0.4, 0.4, size=(n_sources, 3))
dipole_moments = rng.normal(size=(n_sources, 3))

multipole_coefficients = cdfmm.p2m_dipole(
    source_centre,
    source_positions,
    dipole_moments,
    order=expansion_order,
)
local_coefficients = cdfmm.m2l(
    multipole_coefficients,
    source_centre,
    local_centre,
    order=expansion_order,
)

print(f"Expansion order: {expansion_order}")
print(f"Coefficient count: {len(local_coefficients)}")
print(f"Source/local separation: {np.linalg.norm(local_centre - source_centre):.3f}")

## Evaluate a two-dimensional target region

In [ ]:
offset_coordinates = np.linspace(
    -plane_half_width,
    plane_half_width,
    grid_points_per_axis,
)
x_offsets, y_offsets = np.meshgrid(offset_coordinates, offset_coordinates)
target_positions = np.column_stack(
    [
        local_centre[0] + x_offsets.ravel(),
        local_centre[1] + y_offsets.ravel(),
        np.full(x_offsets.size, local_centre[2]),
    ]
)

reference_fields = direct_fields(target_positions, source_positions, dipole_moments)
l2p_fields = local_fields(
    target_positions,
    local_coefficients,
    local_centre,
    expansion_order,
)
m2p_fields = multipole_fields(
    target_positions,
    multipole_coefficients,
    source_centre,
    expansion_order,
)

l2p_errors = relative_error(l2p_fields, reference_fields).reshape(x_offsets.shape)
m2p_errors = relative_error(m2p_fields, reference_fields).reshape(x_offsets.shape)
print("L2P error metrics:", error_metrics(l2p_fields, reference_fields))
print("M2P error metrics:", error_metrics(m2p_fields, reference_fields))

## Spatial error heatmap

In [ ]:
colour_minimum = min(np.log10(l2p_errors).min(), np.log10(m2p_errors).min())
colour_maximum = max(np.log10(l2p_errors).max(), np.log10(m2p_errors).max())

figure, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for current_axes, error_values, title in [
    (axes[0], l2p_errors, "P2M + M2L + L2P"),
    (axes[1], m2p_errors, "P2M + M2P"),
]:
    image = current_axes.imshow(
        np.log10(error_values),
        origin="lower",
        extent=[-plane_half_width, plane_half_width, -plane_half_width, plane_half_width],
        vmin=colour_minimum,
        vmax=colour_maximum,
        cmap="magma",
        aspect="equal",
    )
    current_axes.scatter(0.0, 0.0, marker="x", color="cyan", label="local centre")
    current_axes.set_xlabel("x offset from local centre")
    current_axes.set_ylabel("y offset from local centre")
    current_axes.set_title(title)
    current_axes.legend()

figure.colorbar(image, ax=axes, label=r"$\log_{10}$ relative field error")
figure.suptitle("Spatial accuracy in the local target plane")
figure.tight_layout()

## What to observe

The local Taylor representation is typically best near its expansion centre,
with error increasing toward the region boundary. M2P has a different spatial
error pattern because it expands about the source centre. Both routes invoke
the same C++ coefficients and evaluators used by the operator tests.